# Classroom Lab: API-Based Embeddings with OpenRouter

Analyse 160 Persian Telegram posts with an embedding model called through an API. There is no gold standard: this is an initial research map for interpretation and human review.

## Before you begin

1. Create an account at [OpenRouter](https://openrouter.ai/) and create an API key under **Keys**.
2. Open this file in [Google Colab](https://colab.research.google.com/) using **File → Upload notebook**.
3. Enter your key only in the hidden prompt. Never put it in the notebook, a screenshot, or a shared file.

In [ ]:
# Install packages — run once
!pip -q install pandas scikit-learn matplotlib requests

In [ ]:
# Upload the CSV file from the Student Pack
from google.colab import files
uploaded = files.upload()

In [ ]:
import io
from getpass import getpass
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from sklearn.decomposition import PCA

DATA_FILE = 'telegram_policy_radar_4topics_fa.csv'
df = pd.read_csv(io.BytesIO(uploaded[DATA_FILE]))
print(f'Posts loaded: {len(df)}')
df.head(3)

## Four topic definitions

The API converts every post and every topic definition into a numerical vector. Cosine similarity then assigns each post to its nearest topic. Try editing a definition and observe how the findings change.

In [ ]:
TOPICS = {
    'Regulation and policy': 'News about regulation, licences, rules, public institutions, councils, and digital-economy policy.',
    'Artificial intelligence and data': 'News about artificial intelligence, language models, data, algorithms, and AI applications.',
    'Competition and digital platforms': 'News about competition, monopoly, market power, platforms, e-commerce, and digital firms.',
    'Telecom and digital infrastructure': 'News about the internet, telecom operators, fibre networks, 5G, connectivity, and communications infrastructure.'
}
CODES = {'Regulation and policy': 'REG', 'Artificial intelligence and data': 'AI', 'Competition and digital platforms': 'COMP', 'Telecom and digital infrastructure': 'TELCO'}
EMBEDDING_MODEL = 'openai/text-embedding-3-small'
TOPICS

## Secure API connection

Each student enters their own key. The key is not saved in the notebook or included in the output file.

In [ ]:
OPENROUTER_API_KEY = getpass('OpenRouter API key (hidden): ')
assert OPENROUTER_API_KEY.strip(), 'An API key is required.'
HEADERS = {'Authorization': f'Bearer {OPENROUTER_API_KEY}', 'Content-Type': 'application/json', 'HTTP-Referer': 'https://colab.research.google.com/', 'X-OpenRouter-Title': 'BANA Embeddings Workshop'}
API_URL = 'https://openrouter.ai/api/v1/embeddings'

## Send embedding requests to the API

The posts are sent in batches of 32. Each response returns embedding vectors and token usage.

In [ ]:
def embed_batch(texts):
    response = requests.post(API_URL, headers=HEADERS, json={'model': EMBEDDING_MODEL, 'input': texts, 'encoding_format': 'float'}, timeout=90)
    response.raise_for_status()
    payload = response.json()
    vectors = [item['embedding'] for item in sorted(payload['data'], key=lambda item: item['index'])]
    return vectors, payload.get('usage', {}).get('total_tokens', 0)

def embed_all(texts, batch_size=32):
    vectors, total_tokens = [], 0
    for start in range(0, len(texts), batch_size):
        batch_vectors, batch_tokens = embed_batch(texts[start:start + batch_size])
        vectors.extend(batch_vectors)
        total_tokens += batch_tokens
        print(f'Processed: {min(start + batch_size, len(texts))}/{len(texts)}')
    return np.asarray(vectors, dtype='float32'), total_tokens

## Create embeddings and classify the posts

The API is called for the posts and for the four topic definitions. Note the total token use after execution.

In [ ]:
post_vectors, post_tokens = embed_all(df['text_fa'].tolist())
topic_vectors, topic_tokens = embed_all(list(TOPICS.values()))
post_vectors /= np.linalg.norm(post_vectors, axis=1, keepdims=True)
topic_vectors /= np.linalg.norm(topic_vectors, axis=1, keepdims=True)
scores = post_vectors @ topic_vectors.T
labels = list(TOPICS)
df['predicted_topic'] = [labels[i] for i in scores.argmax(axis=1)]
df['similarity'] = scores.max(axis=1).round(3)
df['review_needed'] = np.where(df['similarity'] < 0.40, 'yes', 'no')
print('Model:', EMBEDDING_MODEL, '| API tokens:', post_tokens + topic_tokens)
df[['student_id', 'predicted_topic', 'similarity', 'review_needed', 'text_fa']].head()

## 1. Topic distribution

Which themes are most common in this sample?

In [ ]:
counts = df['predicted_topic'].value_counts().reindex(labels, fill_value=0)
plt.figure(figsize=(8, 4.5))
plt.bar([CODES[x] for x in labels], counts.values, color=['#5B8FF9', '#61DDAA', '#65789B', '#F6BD16'])
plt.title('Telegram policy radar: predicted themes')
plt.ylabel('Posts')
for i, value in enumerate(counts.values):
    plt.text(i, value + 1, str(value), ha='center')
plt.show()
pd.DataFrame({'topic': labels, 'code': [CODES[x] for x in labels], 'posts': counts.values})

## 2. Embedding map

Each point is one post. PCA compresses the high-dimensional embeddings into two dimensions for visual exploration; it is not a precise measurement of distance.

In [ ]:
coords = PCA(n_components=2, random_state=42).fit_transform(post_vectors)
plt.figure(figsize=(8.5, 6.5))
for code, color, label in zip(['REG', 'AI', 'COMP', 'TELCO'], ['#5B8FF9', '#61DDAA', '#65789B', '#F6BD16'], labels):
    mask = df['predicted_topic'].eq(label)
    plt.scatter(coords[mask, 0], coords[mask, 1], label=code, color=color, alpha=.72, s=34)
plt.title('Telegram policy radar: embedding map (PCA)')
plt.xlabel('PCA dimension 1'); plt.ylabel('PCA dimension 2')
plt.legend(title='Topic code')
plt.show()

## 3. Most representative posts

A high similarity score does not prove the label is correct. It only means the post is closer to that topic definition than to the other definitions.

In [ ]:
for label in labels:
    print('\n' + '=' * 70)
    print(CODES[label], '—', label)
    display(df[df['predicted_topic'].eq(label)].nlargest(3, 'similarity')[['similarity', 'text_fa']])

## 4. Semantic search using the same API

Edit the query below. The API embeds the query and retrieves the five nearest posts.

In [ ]:
QUERY = 'News about regulation and policy for digital platforms'
query_vectors, query_tokens = embed_all([QUERY])
query_vector = query_vectors[0] / np.linalg.norm(query_vectors[0])
df['query_similarity'] = post_vectors @ query_vector
print('Query:', QUERY, '| API tokens:', query_tokens)
display(df.nlargest(5, 'query_similarity')[['query_similarity', 'predicted_topic', 'text_fa']])

## 5. Human review and download

Discuss: Which two topics are dominant? Why might an ambiguous post belong to more than one topic? What human review would you require before using this in a real report?

In [ ]:
review_queue = df.nsmallest(12, 'similarity')[['student_id', 'predicted_topic', 'similarity', 'text_fa']]
display(review_queue)

## 6. Download results

Save the predicted labels, similarity scores, and semantic-search scores for further analysis.

In [ ]:
# Save and download results
OUTPUT_FILE = 'telegram_policy_radar_4topics_openrouter_results.csv'
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
from google.colab import files
files.download(OUTPUT_FILE)